# E-Commerce Sales — Exploratory Data Analysis
## CodeAlpha Data Analytics Internship — Task 2

Complete EDA workflow covering data understanding, cleaning, descriptive statistics, sales, product, category, regional and profitability analysis, correlation analysis, visualizations, and business insights.

**Dataset:** Put the CSV file in the same folder as this notebook. Change `DATA_FILE` below if needed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_FILE = "ecommerce_sales.csv"


In [ ]:
# Load dataset
path = Path(DATA_FILE)
if not path.exists():
    csv_files = list(Path(".").glob("*.csv"))
    if csv_files:
        path = csv_files[0]
        print("Using detected CSV:", path.name)
    else:
        raise FileNotFoundError("Put the e-commerce CSV in the same folder as this notebook.")

df = pd.read_csv(path)
print("Dataset shape:", df.shape)
display(df.head())


## 1. Data Understanding

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))


In [ ]:
# Missing values and duplicates
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"Missing Values": missing, "Missing %": missing_pct})
display(missing_table[missing_table["Missing Values"] > 0])
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# Descriptive statistics
display(df.describe(include="all").T)


## 2. Data Cleaning

In [ ]:
data = df.copy()

# Convert common numeric business columns
for col in ["Sales", "Profit", "Discount", "Quantity", "Amount", "Revenue", "Price", "Cost"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

before = len(data)
data = data.drop_duplicates().copy()
print("Duplicate rows removed:", before - len(data))

# Fill missing numeric values with median
for col in data.select_dtypes(include=np.number).columns:
    data[col] = data[col].fillna(data[col].median())

print("Cleaned shape:", data.shape)


## 3. Sales Analysis

In [ ]:
sales_col = next((c for c in ["Sales", "Revenue", "Amount"] if c in data.columns), None)

if sales_col:
    print("Sales column:", sales_col)
    print("Total Sales:", data[sales_col].sum())
    print("Average Sales:", data[sales_col].mean())

    plt.figure(figsize=(10, 5))
    sns.histplot(data[sales_col], kde=True)
    plt.title("Distribution of Sales")
    plt.xlabel(sales_col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()
else:
    print("No standard Sales/Revenue/Amount column found.")


## 4. Category and Product Analysis

In [ ]:
category_col = next((c for c in ["Category", "Product Category", "Category Name"] if c in data.columns), None)

if sales_col and category_col:
    category_sales = data.groupby(category_col)[sales_col].sum().sort_values(ascending=False)
    display(category_sales.to_frame("Total Sales"))

    plt.figure(figsize=(9, 5))
    category_sales.plot(kind="bar")
    plt.title("Sales by Category")
    plt.xlabel(category_col)
    plt.ylabel("Total Sales")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Category or sales column not found.")


In [ ]:
product_col = next((c for c in ["Product", "Product Name", "Product_Name", "Item"] if c in data.columns), None)

if sales_col and product_col:
    top_products = data.groupby(product_col)[sales_col].sum().sort_values(ascending=False).head(10)
    display(top_products.to_frame("Total Sales"))

    plt.figure(figsize=(10, 6))
    top_products.sort_values().plot(kind="barh")
    plt.title("Top 10 Products by Sales")
    plt.xlabel("Total Sales")
    plt.tight_layout()
    plt.show()
else:
    print("Product or sales column not found.")


## 5. Regional Analysis

In [ ]:
region_col = next((c for c in ["Region", "State", "City", "Country"] if c in data.columns), None)

if sales_col and region_col:
    regional_sales = data.groupby(region_col)[sales_col].sum().sort_values(ascending=False).head(10)
    display(regional_sales.to_frame("Total Sales"))

    plt.figure(figsize=(10, 5))
    regional_sales.sort_values().plot(kind="barh")
    plt.title("Top Regions by Sales")
    plt.xlabel("Total Sales")
    plt.tight_layout()
    plt.show()
else:
    print("Region/location or sales column not found.")


## 6. Profitability Analysis

In [ ]:
profit_col = next((c for c in ["Profit", "Profit/Loss", "Net Profit"] if c in data.columns), None)

if profit_col:
    print("Total Profit:", data[profit_col].sum())
    print("Average Profit:", data[profit_col].mean())

    if category_col:
        category_profit = data.groupby(category_col)[profit_col].sum().sort_values(ascending=False)
        display(category_profit.to_frame("Total Profit"))

        plt.figure(figsize=(9, 5))
        category_profit.plot(kind="bar")
        plt.title("Profit by Category")
        plt.xlabel(category_col)
        plt.ylabel("Total Profit")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print("Profit column not found.")


## 7. Discount vs Profit and Sales vs Profit

In [ ]:
discount_col = next((c for c in ["Discount", "Discount %", "Discount_Percentage"] if c in data.columns), None)

if discount_col and profit_col:
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=data, x=discount_col, y=profit_col)
    plt.title("Discount vs Profit")
    plt.tight_layout()
    plt.show()

if sales_col and profit_col:
    plt.figure(figsize=(9, 5))
    sns.scatterplot(data=data, x=sales_col, y=profit_col)
    plt.title("Sales vs Profit")
    plt.tight_layout()
    plt.show()


## 8. Correlation Analysis

In [ ]:
numeric_data = data.select_dtypes(include=np.number)

if numeric_data.shape[1] >= 2:
    corr = numeric_data.corr()
    plt.figure(figsize=(10, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()
    display(corr)
else:
    print("Not enough numeric columns for correlation analysis.")


## 9. Key Business Insights

After running the notebook, review the charts and tables to identify:
- Highest-sales categories and products
- Most profitable categories and products
- Best-performing regions
- Relationship between discounts and profit
- Relationship between sales and profit
- Areas where pricing, discount, product, or regional strategy could improve

## Conclusion

This project demonstrates how Python-based EDA can transform raw e-commerce sales data into useful business insights for data-driven decision making.